In [ ]:
# Install Java JDK
!apt-get install -y openjdk-17-jdk-headless -qq > /dev/null
!java -version

# Set Java environment variable
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

openjdk version "17.0.17" 2025-10-21
OpenJDK Runtime Environment (build 17.0.17+10-Ubuntu-122.04)
OpenJDK 64-Bit Server VM (build 17.0.17+10-Ubuntu-122.04, mixed mode, sharing)


In [ ]:
%%writefile SSTOAnalysis.java
import java.io.FileWriter;
import java.io.IOException;
import java.io.PrintWriter;

public class SSTOAnalysis {

    static final double G0 = 9.80665;          // standard gravity, m/(s*s)
    static final double DELTA_V = 9400.0;       // required Δv to LEO, m/s

    public static void main(String[] args) {
        String fileName = "ssto_results.csv";
        try (PrintWriter writer = new PrintWriter(new FileWriter(fileName))) {
            // CSV header
            writer.println("Isp_s,epsilon,payloadFraction");

            // Sweep over Isp from 300 to 900 s, step 50 s
            for (int isp = 300; isp <= 900; isp += 50) {
                // Sweep over epsilon from 0.05 to 0.15, step 0.01
                for (double eps = 0.05; eps <= 0.151; eps += 0.01) { // include 0.15
                    double lambda = payloadFraction(DELTA_V, isp, eps);
                    writer.printf("%d,%.3f,%.6f\n", isp, eps, lambda);
                }
            }
            System.out.println("Results written to " + fileName);
        } catch (IOException e) {
            System.err.println("Error writing file: " + e.getMessage());
        }
    }

    /**
     * Computes payload fraction λ for a single stage.
     * @param deltaV required velocity change (m/s)
     * @param Isp specific impulse (s)
     * @param epsilon structural coefficient (0 < ε < 1)
     * @return payload fraction λ (could be negative if impossible)
     */
    public static double payloadFraction(double deltaV, double Isp, double epsilon) {
        double exponent = -deltaV / (Isp * G0);
        double oneOverR = Math.exp(exponent);   // = 1/R
        double numerator = oneOverR - epsilon;
        double denominator = 1.0 - epsilon;
        return numerator / denominator;
    }
}

Writing SSTOAnalysis.java


In [ ]:
!javac SSTOAnalysis.java
print("Compilation successful!")

Compilation successful!


In [ ]:
!java SSTOAnalysis

Results written to ssto_results.csv


In [ ]:
from google.colab import files
files.download('ssto_results.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>